In [ ]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict, List, Optional, Literal
import subprocess
from openai import OpenAI


class Question(TypedDict):
    id: str
    text: str
    choices: List[str]   # ["A) ...", "B) ...", ...]
    correct: str         # "A" ~ "E"
    year: int
    grade: int

class GaussState(TypedDict):
    grade: int
    problem_bank: List[Question]
    current_question: Optional[Question]
    answered_count: int
    correct_count: int
    score: int
    last_answer: Optional[str]

#### Tools
def load_problems_from_past_contests(grade: int) -> List[Question]:
    """
    TODO:
      - CEMC Gauss Past Contests PDF를 파싱해서 JSON/DB로 저장해두고 여기서 로드.
      - 지금은 간단한 예시 dummy 데이터.
    """
    dummy_questions: List[Question] = [
        {
            "id": "2024-G7-Q1",
            "text": "What is 2 + 3?",
            "choices": ["A) 4", "B) 5", "C) 6", "D) 7", "E) 8"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
        {
            "id": "2024-G7-Q2",
            "text": "What is 10 - 4?",
            "choices": ["A) 5", "B) 6", "C) 7", "D) 8", "E) 9"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
    ]
    return dummy_questions




In [ ]:
def init_session(state: GaussState) -> GaussState:
    """
    - 학년(grade)을 이미 state에 넣어두었다고 가정,
      실제 구현에서는 여기서 사용자에게 7/8을 물어보는 로직 추가.
    - 점수 관련 필드를 초기화.
    """
    grade = state.get("grade", 7)  # 디폴트 7학년
    return {
        "grade": grade,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }

def load_past_contests(state: GaussState) -> GaussState:
    """
    - grade에 맞는 과거 Gauss 문제를 로딩하여 problem_bank에 저장.
    """
    grade = state["grade"]
    problems = load_problems_from_past_contests(grade)
    return {
        "problem_bank": problems
    }

def select_question(state: GaussState) -> GaussState:
    """
    - answered_count를 인덱스로 사용해서 다음 문제를 선택.
    - 더 이상 문제가 없으면 current_question을 None으로 둘 수 있음.
    """
    idx = state["answered_count"]
    problem_bank = state["problem_bank"]
    if idx < len(problem_bank):
        q = problem_bank[idx]
    else:
        q = None  # 문제 고갈 (실제 구현에서는 처리 로직 추가)
    return {
        "current_question": q
    }

def ask_question(state: GaussState) -> GaussState:
    """
    - current_question 내용을 사용자에게 보여주고,
    - 사용자의 입력(A~E)을 받아서 last_answer에 채우는 역할.
    - 여기서는 스켈레톤이라, 외부에서 last_answer를 주입하는 구조를 가정.
    - LangGraph에서는 이 노드를 거친 후, 앱 레벨에서 사용자의 답을 받아
      state["last_answer"]를 채워 넣고 다음으로 진행하게 할 수 있음.
    """
    q = state["current_question"]
    if q is None:
        print("No more questions available.")
    else:
        print("Question:", q["text"])
        for choice in q["choices"]:
            print(choice)
        print("Please answer with one of: A, B, C, D, E")
    # 이 노드는 상태를 바꾸지 않고, 단순히 질문만 보여준다고 가정
    return {}

def check_answer(state: GaussState) -> GaussState:
    """
    - last_answer와 current_question.correct를 비교하여 채점.
    - 점수 및 카운터 업데이트.
    """
    q = state["current_question"]
    user_answer = (state.get("last_answer") or "").strip().upper()

    if q is None:
        # 더 이상 문제가 없을 때의 처리 (여기서는 상태 변화 없음)
        return {}

    is_correct = user_answer == q["correct"]

    answered_count = state["answered_count"] + 1
    correct_count = state["correct_count"] + (1 if is_correct else 0)
    # 초기 버전: 맞으면 1점, 틀리면 0점
    score = state["score"] + (1 if is_correct else 0)

    # 간단 피드백 (실전에서는 LLM으로 자연어 설명 생성 가능)
    if is_correct:
        print("Correct! 🎉")
    else:
        print(f"Incorrect. The correct answer was {q['correct']}.")

    return {
        "answered_count": answered_count,
        "correct_count": correct_count,
        "score": score,
    }

def decide_continue(state: GaussState) -> Literal["continue", "stop"]:
    """
    - 간단 조건:
      - 10문제 미만이면 계속,
      - 아니면 종료.
    - 또는 문제 고갈 시 종료하는 조건 등을 추가할 수 있음.
    """
    max_questions = 10
    if state["answered_count"] >= max_questions:
        return "stop"

    # 문제 고갈 체크
    if state["answered_count"] >= len(state["problem_bank"]):
        return "stop"

    return "continue"

def end_session(state: GaussState) -> GaussState:
    """
    - 세션 요약 출력.
    """
    answered = state["answered_count"]
    correct = state["correct_count"]
    score = state["score"]
    accuracy = (correct / answered * 100) if answered > 0 else 0.0

    print("=== Session Summary ===")
    print(f"Questions answered: {answered}")
    print(f"Correct answers:    {correct}")
    print(f"Score (simple):     {score}")
    print(f"Accuracy:           {accuracy:.1f}%")

    return {}



In [ ]:
def build_gauss_graph():
    builder = StateGraph(GaussState)  # Python API 기준[web:36][web:43]

    # 노드 추가
    builder.add_node("init_session", init_session)
    builder.add_node("load_past_contests", load_past_contests)
    builder.add_node("select_question", select_question)
    builder.add_node("ask_question", ask_question)
    builder.add_node("check_answer", check_answer)
    builder.add_node("end_session", end_session)

    # 엣지 연결
    builder.add_edge(START, "init_session")
    builder.add_edge("init_session", "load_past_contests")
    builder.add_edge("load_past_contests", "select_question")
    builder.add_edge("select_question", "ask_question")
    builder.add_edge("ask_question", "check_answer")

    # 조건부 엣지 (라우터)
    builder.add_conditional_edges(
        "check_answer",
        decide_continue,           # router 함수
        {
            "continue": "select_question",
            "stop": "end_session",
        },
    )

    builder.add_edge("end_session", END)

    graph = builder.compile()
    return graph


In [ ]:
if __name__ == "__main__":
    graph = build_gauss_graph()

    # 초기 상태에서 grade만 지정해주고 시작 (예: 7학년)
    state: GaussState = {
        "grade": 7,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }

    # 간단한 루프 예시 (실전에서는 LangGraph의 streaming/async 등을 활용 가능)[web:11][web:43]
    while True:
        result = graph.invoke(state)
        state.update(result)

        # 그래프가 한 번 돌고 나면, ask_question → check_answer까지 지나간 상태가 됨.
        # 여기서 사용자 입력을 받아 last_answer를 업데이트해주고, 다시 invoke 하는 식으로 만들 수도 있음.
        if state["answered_count"] >= 10 or state["answered_count"] >= len(state["problem_bank"]):
            break

        # 여기서는 그냥 디폴트로 항상 "B"라고 답하는 더미 입력 예시
        state["last_answer"] = "B"
